# 7장 학습 루프: 이 PC 에서 몇 시간 안에 (실습)

교재 `docs/book/07-training.md` 와 함께 본다. 이 노트북에서 하는 것:

1. `Trainer` 로 짧은 학습(600 스텝, 약 3분)을 돌리며 학습률 스케줄·평가·로그·체크포인트가 어떻게 남는지 본다
2. 긴 학습(`scripts/train.py`, 수 시간)의 로그를 읽어 손실 곡선을 그린다. 과적합이 시작되는 지점
3. 체크포인트를 되살려 생성해 본다
4. 3장에서 세어서 만든 임베딩 대신 **학습된** 토큰 임베딩의 이웃을 본다

> 전체 실행 약 4분 (긴 학습은 노트북 밖에서 별도 실행).

## 1. 짧은 학습. Trainer 한 바퀴

In [ ]:
import time

import torch

from shllm.config import RUNS_DIR, TOKENIZER_DIR, setup_cpu
from shllm.data import load_corpus
from shllm.model import GPT
from shllm.tokenizer import BPETokenizer
from shllm.train import Trainer, load_yaml_config, lr_at

setup_cpu()
model_cfg, train_cfg = load_yaml_config("../configs/tiny-notebook.yaml")
print(model_cfg)
print(train_cfg)

In [ ]:
tok = BPETokenizer.load(TOKENIZER_DIR / train_cfg.tokenizer)
data = torch.tensor(tok.encode(load_corpus("korean-classics")))
n = int(0.9 * len(data))
train, val = data[:n], data[n:]

model = GPT(model_cfg)
print(f"{model.n_params():,} 파라미터, 스텝당 {train_cfg.batch_size * model_cfg.block_size:,} 토큰")
trainer = Trainer(model, train, val, train_cfg)
t0 = time.perf_counter()
history = trainer.train()
print(f"{time.perf_counter() - t0:.0f}초, best val {trainer.best_val:.3f}")

**출력에서 볼 것**: 스텝 0 의 loss 9.03 ≈ log 8192. 100 스텝마다 train·val 이 같이 내려가고 마지막에 `best val` 이 기록된다. 80초에 600 스텝이면 스텝당 0.13초, 6.8M 모델(small-cpu)은 이 5배 크기·2배 배치라 스텝당 약 1초다.

**해 보기**: `train_cfg.lr = 1e-2` 로 열 배 올려 다시 돌리면 발산하는지, `warmup_steps = 0` 이면 초기 곡선이 어떻게 달라지는지 보라 (`Trainer(...)` 를 새로 만들어야 한다).

### 1.1 무엇이 남았나

In [ ]:
run = RUNS_DIR / train_cfg.run_name
print("로그  :", run / "log.jsonl", "→", len(history), "줄")
print("설정  :", (run / "config.yaml").read_text()[:120].replace("\n", " | "), "...")
print("체크포인트:", sorted(p.name for p in trainer.ckpt_dir.iterdir()))
print(history[-1])

### 1.2 학습률 스케줄

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager

cjk = [f.name for f in font_manager.fontManager.ttflist if "CJK" in f.name]
if cjk:
    matplotlib.rcParams["font.family"] = cjk[0]

steps = list(range(train_cfg.max_steps + 1))
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(steps, [lr_at(s, train_cfg) for s in steps])
axes[0].set_title("학습률: warmup → cosine → min_lr")
axes[0].set_xlabel("스텝")
axes[1].plot([h["step"] for h in history], [h["train_loss"] for h in history], "o-", label="train")
axes[1].plot([h["step"] for h in history], [h["val_loss"] for h in history], "s-", label="val")
axes[1].set_title("짧은 학습의 손실")
axes[1].set_xlabel("스텝")
axes[1].legend()
plt.show()

warmup 은 처음 몇 스텝을 작은 학습률로 시작해 랜덤 초기 상태에서 큰 걸음을 밟지 않게 한다. 그 뒤 코사인으로 부드럽게 줄여 마지막엔 세밀하게 조정한다.

## 2. 긴 학습의 손실 곡선

`uv run python scripts/train.py --config configs/small-cpu.yaml` 로 노트북 밖에서 돌린 run 의 로그를 읽는다.
(아직 돌리지 않았거나 진행 중이면 그때까지의 곡선이 그려진다.)

In [ ]:
from shllm.train import read_log

log = read_log("small-cpu")
if not log:
    print("data/runs/small-cpu/log.jsonl 이 없습니다. scripts/train.py 를 먼저 실행하세요")
else:
    best = min(log, key=lambda r: r["val_loss"])
    print(f"{len(log)} 평가 시점, 마지막 step {log[-1]['step']}, 경과 {log[-1]['elapsed'] / 60:.0f}분")
    print(f"best val {best['val_loss']:.3f} @ step {best['step']}   (마지막 train {log[-1]['train_loss']:.3f} / val {log[-1]['val_loss']:.3f})")
    fig, ax = plt.subplots(figsize=(7, 3.8))
    ax.plot([r["step"] for r in log], [r["train_loss"] for r in log], label="train")
    ax.plot([r["step"] for r in log], [r["val_loss"] for r in log], label="val")
    ax.axvline(best["step"], ls="--", c="gray", label=f"best val @ {best['step']}")
    ax.set_xlabel("스텝")
    ax.set_ylabel("loss (nat/토큰)")
    ax.set_title("small-cpu: 6층 · C=256 · 문맥 128")
    ax.legend()
    plt.show()

train 은 계속 내려가고 val 은 어느 지점에서 멈추거나 오른다. 1장 n-gram, 4장 MLP 에서 본 **과적합**이 GPT 에서도 똑같이 일어난다.
코퍼스가 49만 토큰뿐이라 20 에폭이면 모델이 문장을 외우기 시작한다. `best.pt` 는 val 이 가장 낮았던 시점이고, 8장은 그것을 쓴다.
근본적인 해법은 **데이터를 늘리는 것**이다(코퍼스 2차 확장, todo).

## 3. 체크포인트에서 생성

In [ ]:
from shllm.generate import generate_text
from shllm.train import load_checkpoint

for run_name in ("small-cpu", train_cfg.run_name):
    try:
        m, ck = load_checkpoint(run_name, "best.pt")
    except FileNotFoundError:
        print(f"[{run_name}] 체크포인트 없음")
        continue
    print(f"[{run_name}] step {ck['step']}, {m.n_params():,} 파라미터")
    torch.manual_seed(7)
    print(generate_text(m, tok, "옛날 옛적에 호랑이가", max_new_tokens=80, temperature=0.8, top_k=50))
    print()

## 4. 학습된 임베딩: 3장 그림 다시 그리기

3장에서는 세어서(PPMI+SVD) 벡터를 만들었다. 학습된 GPT 의 토큰 임베딩 표 `(V, C)` 로 같은 이웃 찾기를 한다.

In [ ]:
from shllm.embedding import nearest

try:
    m, _ = load_checkpoint("small-cpu", "best.pt")
except FileNotFoundError:
    m = trainer.model
E = m.embed.tok.weight.detach()  # (V, C) 학습된 표
for w in (" 아버지", " 돈", " 서울", " 없다", "는"):
    i = tok.encode(w)
    if len(i) != 1:
        continue
    print(f"{w!r:>8} →", [(tok.token_str(j), round(s, 2)) for j, s in nearest(E, i[0], k=6)])

In [ ]:
# 3장의 그림을 학습된 임베딩으로: 같은 단어 무리를 2차원(PCA)으로
groups = {
    "가족": [" 아버지", " 어머니", " 아들", " 딸", " 아내", " 남편"],
    "돈·숫자": [" 돈", " 원", " 백", " 천", " 만", " 오백"],
    "조사·어미": ["는", "가", "를", "고", "었다", "는다"],
    "장소": [" 서울", " 시골", " 집", " 학교", " 방", " 병원"],
}
present = {g: [w for w in ws if len(tok.encode(w)) == 1] for g, ws in groups.items()}
sel = [w for ws in present.values() for w in ws]
X = E[[tok.encode(w)[0] for w in sel]]
X = X - X.mean(0)
_, _, Vt = torch.linalg.svd(X, full_matrices=False)
P = X @ Vt[:2].T
fig, ax = plt.subplots(figsize=(7, 5.5))
i = 0
for g, ws in present.items():
    pts = P[i : i + len(ws)]
    ax.scatter(pts[:, 0], pts[:, 1], label=g, s=40)
    for w, (px, py) in zip(ws, pts):
        ax.annotate(w.strip(), (px, py), fontsize=9, xytext=(3, 3), textcoords="offset points")
    i += len(ws)
ax.set_title("학습된 토큰 임베딩 (C=256) → 2차원 PCA")
ax.legend()
plt.show()

학습된 표도 같은 성질을 보인다. 아무도 뜻을 가르치지 않았지만 "다음 토큰을 잘 맞히는" 목적이 비슷한 토큰을 비슷한 벡터로 만든다.
(짧은 학습 모델이면 이웃이 거칠다. 긴 학습 후 다시 실행해 보라.)

## 정리

- 학습 루프 = 배치 → forward → loss → backward → clip → AdamW step, 그 위에 warmup+cosine 스케줄·주기적 평가·JSONL 로그·체크포인트.
- 로그와 체크포인트는 `data/runs/<run>/`, `data/checkpoints/<run>/` 규약을 따른다. 8·9장과 대시보드가 이 파일을 읽는다.
- 코퍼스가 작으면 과적합이 빨리 온다: dropout·weight decay·early best 로 버티되, 근본 해법은 데이터.
- CPU 에서는 스레드 수(물리 코어)와 스텝당 토큰 수가 속도를 정한다.

---
**다음 장**: 8장, 확률에서 문장을 뽑는 방법: temperature · top-k · top-p · 반복 억제.